In [2]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_groq import ChatGroq

# Load environment variables
load_dotenv(find_dotenv())


True

In [3]:
# ----------------------------------------------------------------------
# Alternative: Using init_chat_model from langchain.chat_models
# ----------------------------------------------------------------------
from langchain.chat_models import init_chat_model

# init_chat_model is a utility function in LangChain designed to initialize chat models from any supported provider using a unified interface.
# Instead of importing distinct provider-specific classes (like ChatOpenAI, ChatAnthropic, or ChatGroq), you can instantiate a model by passing its name and provider as a string or prefix.

chat_via_init = init_chat_model(
    model="openai/gpt-oss-120b",
    model_provider="groq",
    temperature=0.7
)

response = chat_via_init.invoke("Why do parrots talk?")
response.content


'Parrots don’t “talk” in the same way humans do—they’re amazing mimics. Here’s why they can repeat words and sounds:\n\n| Reason | What it means | How it shows up in a pet parrot |\n|--------|---------------|--------------------------------|\n| **Vocal learning** | Parrots belong to a rare group of animals (along with songbirds, hummingbirds, some mammals, and humans) that can learn new sounds by listening to them. | A newly‑hatched cockatoo will copy the whistling of a nearby toy, then later the phrases you say to it. |\n| **Social bonding** | In the wild, parrots live in tight‑knit flocks. They use calls to stay together, recognize individuals, and reinforce relationships. | A pet parrot may repeat “hello” because it’s a way of getting attention and “talking” with its human “flock.” |\n| **Brain structure** | Their forebrain (the “song system”) is highly developed for processing and reproducing complex acoustic patterns. | This neural circuitry lets them store a wide repertoire of so

In [4]:
# Use .stream() instead of .invoke() to stream token by token
for chunk in chat_via_init.stream("Why do parrots have colorful feathers?"):
    print(chunk.content, end="", flush=True)


Parrots are famous for their bright, kaleidoscopic plumage, and scientists think the colors serve several purposes that have been shaped by evolution over millions of years. Here’s a rundown of the main reasons:

| Function | How it works | Why it matters for parrots |
|----------|--------------|---------------------------|
| **Sexual selection** | Bright colors can signal good health, good genes, and overall vigor. In many bird species, mates prefer partners with the most vivid plumage because it often means they are free of parasites and have a strong immune system. | In many parrot species, both males and females are colorful, but the intensity can still influence mate choice. A brighter feather pattern may give an individual an edge in attracting a partner. |
| **Species and individual identification** | Each parrot species (and sometimes even individual birds) has a distinct color pattern that helps birds recognize conspecifics—members of the same species—especially in dense fores

In [5]:
responses = chat_via_init.batch([
                                "Why do parrots have colorful feathers?",
                                "How do airplanes fly?",
                                "What is quantum computing?"
], config={'max_concurrency': 5}) # At most 5 inputs should be processed concurrently.
for response in responses:
    print(response)

content='**Short answer:**  \nParrots are colorful because their bright plumage gives them evolutionary advantages—helping them attract mates, recognize members of their own species, stay hidden among the dappled light of the forest canopy, and even signal health or warning. The colors come from a mix of special pigments (psittacofulvins) that parrots produce themselves and microscopic feather structures that scatter light.\n\n---\n\n## 1. Evolutionary “why” – the selective pressures behind the rainbow\n\n| Function | How bright feathers help | Evidence / examples |\n|----------|--------------------------|---------------------|\n| **Sexual selection** | Vivid plumage is a visual cue of genetic quality, age, and parasite resistance. Birds that can grow and maintain bright feathers are often healthier, so mates prefer them. | In many parrot species (e.g., *Ara* macaws, *Melopsittacus undulatus*—budgerigars), males are slightly more colorful than females, and courtship displays involve sh

In [6]:
import asyncio

async def main():
    response = await chat_via_init.ainvoke("Hello, how are you?") # Single Async call
    print("ainvoke:", response.content[:50]) # when you need to wait for single response

    responses = await chat_via_init.abatch([            # Process multiple inputs asynchronously in parallel
            "Why do parrots have colorful feathers?",
            "How do airplanes fly?",
            "What is quantum computing?",
            ])
    for r in responses:
        print("abatch:", r.content[:50])

    async for chunk in chat_via_init.astream("Tell me a one-line fact about the moon."):
        print(chunk.content, end="|", flush=True)

# In Jupyter, an event loop is already running, so use 'await' instead of 'asyncio.run()'
await main()


ainvoke: Hello! I'm doing great, thank you for asking. How 
abatch: Parrots are famous for their brilliant reds, blues
abatch: **Short answer:**  
Airplanes stay aloft because t
abatch: **Quantum computing** is a type of computation tha


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m0754txtetft6mp668qjxk9r` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 6182, Requested 2335. Please try again in 3.8775s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}